# 11. Seed-Ensemble Calibration Search (Data-Level Ensemble)

## 목표
- `10_third_dataset_search`의 Stage-1 결과를 기반으로 상위 static 모델 3개를 선택
- 각 모델에 대해 여러 seed에서 캘리브레이션 샘플을 생성하고 **하나의 캘리브레이션 데이터셋으로 병합**
- 병합 데이터로 단일 FP8 static 모델을 생성해 성능/안정성 검증

## 배경 반영
- 기존 `Cell 15`에서 `ENSEMBLE_BASES`가 1개만 잡혀 불일치했던 문제를 회피하기 위해,
  이 노트북은 Stage-1 결과에서 top-k를 자동 선택하는 흐름을 기본으로 사용
- 기본 top-k는 3개이며, 필요하면 수동 override 가능

---
## 0. 공통 설정

In [ ]:
import os
import gc
import re
import json
import glob
import time
import shutil
import random
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

ROOT = Path('${PROJECT_ROOT}')
BASE_MODEL = ROOT / 'open/base_model'
BASE_DIR = ROOT / '11_seed_ensemble_calibration_search/code'
EVAL_SCRIPT = ROOT / 'eval/run_eval.py'
EVAL_OUTPUT_DIR = ROOT / '11_seed_ensemble_calibration_search/eval_results'

SRC_STAGE1_CODE_DIR = ROOT / '10_third_dataset_search/code'
SRC_EVAL_DIRS = [ROOT / '10_third_dataset_search/eval_results', ROOT / 'eval']

SEED = 42
DTYPE = torch.float16
BASE_IGNORE = ['lm_head']
MAX_SEQ_LENGTH = 2048

# Runtime safety defaults (override in later cells)
SAFE_MODE = True
SAFE_STAGE2_TOP_K = 1
SAFE_FORCE_BASE = 'exp_b1_fp8_static_m224_p224_a064_t512'  # 예: 'exp_a0_fp8_static_m256_p256_a000_t512'
CALIB_MAP_NUM_PROC = 1
CALIB_MAX_SEQ_LENGTH = 1536
STAGE2_COOLDOWN_SEC = 12

BASE_DIR.mkdir(parents=True, exist_ok=True)
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'BASE_MODEL: {BASE_MODEL}')
print(f'BASE_DIR:   {BASE_DIR}')
print(f'EVAL_OUT:   {EVAL_OUTPUT_DIR}')
print(f'SRC_CODE:   {SRC_STAGE1_CODE_DIR}')
print(f'SRC_EVAL:   {[str(x) for x in SRC_EVAL_DIRS]}')
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")

In [ ]:
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_model_and_tokenizer():
    model = AutoModelForCausalLM.from_pretrained(
        str(BASE_MODEL),
        torch_dtype=DTYPE,
        device_map='auto',
        trust_remote_code=True,
    )
    tok = AutoTokenizer.from_pretrained(str(BASE_MODEL), trust_remote_code=True)
    return model, tok


def cleanup_model(model=None):
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_model(model, tok, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(out_dir))
    tok.save_pretrained(str(out_dir))

    src_template = BASE_MODEL / 'chat_template.jinja'
    if src_template.exists():
        shutil.copy2(src_template, out_dir / 'chat_template.jinja')

    for req in ['config.json', 'tokenizer.json', 'tokenizer_config.json']:
        assert (out_dir / req).exists(), f'{req} missing: {out_dir}'


def make_submission_zip(model_dir: Path, zip_name: str):
    zip_path = shutil.make_archive(
        base_name=str(model_dir.parent / zip_name),
        format='zip',
        root_dir=str(model_dir.parent),
        base_dir='model',
    )
    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f'  ZIP: {zip_path} ({size_mb:.1f} MB)')
    return zip_path


def compute_competition_score(perf_model, perf_base, t_model, t_base):
    perf_norm = perf_model / perf_base
    speed_norm = 1.0 - (t_model / t_base)
    score_proxy = max(0.5 * perf_norm + 0.5 * speed_norm, 0.0)
    return perf_norm, speed_norm, score_proxy


def _real(p):
    return os.path.realpath(str(p))


def _is_num(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


def _fmt(v, d=4, signed=False):
    if v is None:
        return 'NA'
    return format(v, f'+.{d}f' if signed else f'.{d}f')

---
## 1. Stage-1 기준 실험 정의(10_와 동일)

In [ ]:
# 10_ 실험 정의를 그대로 반영 (ratio/total/pile_mode)
STAGE1_CONFIGS = [
    {'id': 'B0', 'name': 'exp0_fp8_dynamic_base', 'scheme': 'FP8_DYNAMIC', 'manta_n': 0,   'pile_n': 0,   'arc_n': 0,   'pile_mode': 'basic'},
    {'id': 'A0', 'name': 'exp_a0_fp8_static_m256_p256_a000_t512', 'scheme': 'FP8', 'manta_n': 256, 'pile_n': 256, 'arc_n': 0,   'pile_mode': 'basic'},
    {'id': 'A1', 'name': 'exp_a1_fp8_static_m256_p224_a032_t512', 'scheme': 'FP8', 'manta_n': 256, 'pile_n': 224, 'arc_n': 32,  'pile_mode': 'basic'},
    {'id': 'A2', 'name': 'exp_a2_fp8_static_m256_p192_a064_t512', 'scheme': 'FP8', 'manta_n': 256, 'pile_n': 192, 'arc_n': 64,  'pile_mode': 'basic'},
    {'id': 'A3', 'name': 'exp_a3_fp8_static_m256_p160_a096_t512', 'scheme': 'FP8', 'manta_n': 256, 'pile_n': 160, 'arc_n': 96,  'pile_mode': 'basic'},
    {'id': 'B1', 'name': 'exp_b1_fp8_static_m224_p224_a064_t512', 'scheme': 'FP8', 'manta_n': 224, 'pile_n': 224, 'arc_n': 64,  'pile_mode': 'basic'},
    {'id': 'B2', 'name': 'exp_b2_fp8_static_m288_p160_a064_t512', 'scheme': 'FP8', 'manta_n': 288, 'pile_n': 160, 'arc_n': 64,  'pile_mode': 'basic'},
]
STAGE1_MAP = {x['name']: x for x in STAGE1_CONFIGS}
print('Stage-1 configs:', len(STAGE1_CONFIGS))
for x in STAGE1_CONFIGS:
    print('-', x['name'], '|', x['scheme'], '|', (x['manta_n'], x['pile_n'], x['arc_n']))

---
## 2. Stage-1 결과 로드 및 Top-3 선택 (불일치 문제 반영)

In [ ]:
def _iter_eval_files(eval_dirs):
    files = []
    for d in eval_dirs:
        d = Path(d)
        if d.exists():
            files.extend(d.glob('comparison_*.json'))
            files.extend(d.glob('eval_*.json'))

    # nohup 로그에서 저장 경로 추출 fallback
    for d in eval_dirs:
        d = Path(d)
        if not d.exists():
            continue
        for logf in sorted(d.glob('nohup*.log'), key=lambda x: x.stat().st_mtime, reverse=True)[:8]:
            try:
                txt = logf.read_text(encoding='utf-8', errors='ignore')
            except Exception:
                continue
            for m in re.findall(r'결과 저장:\s*(\S*(?:comparison|eval)_\d+\.json)', txt):
                mp = Path(m)
                if mp.exists():
                    files.append(mp)

    uniq = {f.resolve() for f in files if f.exists()}
    return sorted(uniq, key=lambda p: p.stat().st_mtime, reverse=True)


def _load_rows(path: Path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            rows = json.load(f)
        if isinstance(rows, list):
            return rows
    except Exception:
        pass
    return None


def load_best_stage1_summary(eval_dirs, stage1_names):
    # code__ prefix와 raw name 모두 매칭
    target_names = set()
    for n in stage1_names:
        target_names.add(n)
        target_names.add('code__' + n)

    best = None  # (overlap, score_rows, mtime, file, rows)

    for f in _iter_eval_files(eval_dirs):
        rows = _load_rows(f)
        if rows is None:
            continue

        names = {r.get('model_name') for r in rows if isinstance(r, dict)}
        overlap = len(target_names & names)
        score_rows = sum(1 for r in rows if isinstance(r, dict) and _is_num(r.get('score_proxy')))
        score = (overlap, score_rows, f.stat().st_mtime)

        if best is None or score > (best[0], best[1], best[2]):
            best = (overlap, score_rows, f.stat().st_mtime, f, rows)

    if best is None:
        return None, None, 0
    return best[3], best[4], best[0]


def rank_stage1_static(rows):
    ranked = []
    for r in rows:
        name = r.get('model_name', '')
        if not isinstance(name, str):
            continue
        raw = name.replace('code__', '', 1)
        cfg = STAGE1_MAP.get(raw)
        if cfg is None:
            continue
        if cfg['scheme'] != 'FP8':
            continue

        score = r.get('score_proxy')
        perf = r.get('overall_median')
        t = (r.get('time_stats') or {}).get('median')
        if score is None and _is_num(perf) and _is_num(t):
            # score_proxy가 없는 경우 base 계산 fallback
            pass

        ranked.append({
            'raw_name': raw,
            'model_name': name,
            'score_proxy': r.get('score_proxy'),
            'overall': r.get('overall_median'),
            'time': (r.get('time_stats') or {}).get('median'),
            'gsm8k': r.get('gsm8k_median'),
            'mmlu': r.get('mmlu_median'),
            'arc': r.get('arc_challenge_median'),
        })

    # score_proxy 우선, 동점이면 overall, 다음 빠른 시간
    def keyf(x):
        s = x['score_proxy'] if _is_num(x['score_proxy']) else -1e9
        o = x['overall'] if _is_num(x['overall']) else -1e9
        t = x['time'] if _is_num(x['time']) else 1e18
        return (s, o, -t)

    ranked.sort(key=keyf, reverse=True)
    return ranked


TOPK_STATIC = 3
FORCE_TOP_MODELS = []  # 예: ['exp_a0_fp8_static_m256_p256_a000_t512','exp_a1_fp8_static_m256_p224_a032_t512','exp_b1_fp8_static_m224_p224_a064_t512']

summary_file, stage1_rows, overlap = load_best_stage1_summary(SRC_EVAL_DIRS, [x['name'] for x in STAGE1_CONFIGS])
if summary_file is None:
    raise RuntimeError('Stage-1 summary 파일을 찾지 못했습니다. SRC_EVAL_DIRS를 확인하세요.')

print('stage1 summary:', summary_file)
print('name overlap:', overlap)

ranked_static = rank_stage1_static(stage1_rows)
print('\n[Stage-1 static rank]')
for i, r in enumerate(ranked_static, 1):
    print(f"{i:>2d}. {r['raw_name']:<42} score={_fmt(r['score_proxy'])} overall={_fmt(r['overall'])} time={_fmt(r['time'], d=1)}")

if FORCE_TOP_MODELS:
    selected_top = FORCE_TOP_MODELS
else:
    selected_top = [r['raw_name'] for r in ranked_static[:TOPK_STATIC]]

print('\n[selected for seed-ensemble]')
for n in selected_top:
    print('-', n)

# 이전 불일치 이슈 점검: a0,a1이 포함되어야 함(현재 결과 기준)
must_include = {'exp_a0_fp8_static_m256_p256_a000_t512', 'exp_a1_fp8_static_m256_p224_a032_t512'}
missing = [x for x in must_include if x not in selected_top]
if missing:
    print('\n[WARN] top selection does not include expected models:', missing)
    print('FORCE_TOP_MODELS로 수동 고정하세요.')

---
## 3. 데이터 유틸: MANTA + pile + ARC + Seed 병합

In [ ]:
_PILE_CACHE = None
_ARC_CACHE = None


def prepare_manta(tokenizer, n: int, seed: int):
    if n <= 0:
        return None
    ds = load_dataset('LGAI-EXAONE/MANTA-1M', split='train')
    ds = ds.shuffle(seed=seed).select(range(n))

    def _pp(ex):
        return {
            'text': tokenizer.apply_chat_template(
                ex['conversations'],
                add_generation_prompt=True,
                tokenize=False,
            )
        }

    ds = ds.map(_pp, num_proc=CALIB_MAP_NUM_PROC, remove_columns=ds.column_names)
    return ds


def load_pile_raw():
    global _PILE_CACHE
    if _PILE_CACHE is None:
        _PILE_CACHE = load_dataset('NeelNanda/pile-10k', split='train')
    return _PILE_CACHE


def sample_pile(n: int, seed: int, mode: str = 'basic'):
    if n <= 0:
        return None

    raw = load_pile_raw()
    if mode == 'basic':
        return raw.shuffle(seed=seed).select(range(n)).select_columns(['text'])

    if mode == 'exclude_pilecc':
        keep = []
        for i, ex in enumerate(raw):
            meta = ex.get('meta') or {}
            src = meta.get('pile_set_name', '') if isinstance(meta, dict) else ''
            if src != 'Pile-CC':
                keep.append(i)
        rng = random.Random(seed)
        rng.shuffle(keep)
        return raw.select(keep[:n]).select_columns(['text'])

    if mode == 'stratified':
        by_source = {}
        for i, ex in enumerate(raw):
            meta = ex.get('meta') or {}
            src = meta.get('pile_set_name', 'unknown') if isinstance(meta, dict) else 'unknown'
            by_source.setdefault(src, []).append(i)

        total = sum(len(v) for v in by_source.values())
        rng = random.Random(seed)
        selected = []
        for _, idxs in by_source.items():
            k = int(round(n * (len(idxs) / total)))
            if k <= 0:
                continue
            idxs = idxs[:]
            rng.shuffle(idxs)
            selected.extend(idxs[:k])

        if len(selected) < n:
            all_idx = list(range(len(raw)))
            rng.shuffle(all_idx)
            selected.extend(all_idx[:(n - len(selected))])

        return raw.select(selected[:n]).select_columns(['text'])

    raise ValueError(f'unknown pile mode: {mode}')


def load_arc_raw():
    global _ARC_CACHE
    if _ARC_CACHE is None:
        _ARC_CACHE = load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='train')
    return _ARC_CACHE


def prepare_arc(tokenizer, n: int, seed: int):
    if n <= 0:
        return None

    ds = load_arc_raw().shuffle(seed=seed).select(range(n))

    def _pp(ex):
        choices = ex.get('choices', {})
        labels = choices.get('label', []) if isinstance(choices, dict) else []
        texts = choices.get('text', []) if isinstance(choices, dict) else []
        option_lines = [f"{l}. {t}" for l, t in zip(labels, texts)]
        prompt = (
            "다음 객관식 문제를 읽고 정답을 고르세요.\n"
            f"문제: {ex.get('question', '')}\n"
            "선택지:\n" + "\n".join(option_lines)
        )
        msgs = [{'role': 'user', 'content': prompt}]
        text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        return {'text': text}

    ds = ds.map(_pp, num_proc=CALIB_MAP_NUM_PROC, remove_columns=ds.column_names)
    return ds


def build_single_seed_calib(tokenizer, manta_n, pile_n, arc_n, seed, pile_mode='basic'):
    chunks = []

    m = prepare_manta(tokenizer, manta_n, seed)
    if m is not None:
        chunks.append(m)

    p = sample_pile(pile_n, seed, pile_mode)
    if p is not None:
        chunks.append(p)

    a = prepare_arc(tokenizer, arc_n, seed)
    if a is not None:
        chunks.append(a)

    if not chunks:
        raise ValueError('empty calib')

    if len(chunks) == 1:
        ds = chunks[0]
    else:
        ds = concatenate_datasets(chunks)

    return ds.shuffle(seed=seed)


def build_seed_ensemble_calib(
    tokenizer,
    manta_n,
    pile_n,
    arc_n,
    seeds,
    pile_mode='basic',
    ensemble_cap=None,
    final_shuffle_seed=42,
):
    # 각 seed에서 동일 설정으로 calib을 만들고 concat -> (옵션) cap
    parts = []
    for sd in seeds:
        ds = build_single_seed_calib(
            tokenizer=tokenizer,
            manta_n=manta_n,
            pile_n=pile_n,
            arc_n=arc_n,
            seed=sd,
            pile_mode=pile_mode,
        )
        parts.append(ds)

    merged = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    merged = merged.shuffle(seed=final_shuffle_seed)

    if ensemble_cap is not None and len(merged) > ensemble_cap:
        merged = merged.select(range(ensemble_cap))

    return merged

---
## 4. Seed-Ensemble 실험 정의

In [ ]:
# Stage-2 안전 실행 설정
# - SAFE_MODE=True면 Stage-2를 상위 일부만 실행해 커널 dead 리스크를 낮춤
# - SAFE_FORCE_BASE를 지정하면 해당 모델 1개만 Stage-2 실행

print('selected_top (from Stage-1):', selected_top)

if SAFE_MODE:
    if SAFE_FORCE_BASE is not None:
        stage2_target_names = [SAFE_FORCE_BASE]
    else:
        stage2_target_names = selected_top[:SAFE_STAGE2_TOP_K]
else:
    stage2_target_names = selected_top

print('SAFE_MODE:', SAFE_MODE)
print('CALIB_MAP_NUM_PROC:', CALIB_MAP_NUM_PROC)
print('CALIB_MAX_SEQ_LENGTH:', CALIB_MAX_SEQ_LENGTH)
print('STAGE2_COOLDOWN_SEC:', STAGE2_COOLDOWN_SEC)
print('stage2_target_names:', stage2_target_names)


In [ ]:
ENSEMBLE_SEEDS = [13, 42, 52, 77, 101]
ENSEMBLE_CAP = 1024  # None이면 full concat(= base_total * seed_count)
RUN_STAGE2 = True   # 실행 직전에 True로 변경

if 'stage2_target_names' not in globals():
    stage2_target_names = selected_top

EXPERIMENTS_STAGE2 = []
for name in stage2_target_names:
    cfg = STAGE1_MAP[name]
    cap_tag = 'full' if ENSEMBLE_CAP is None else str(ENSEMBLE_CAP)
    exp_name = f"exp_se_{name}_k{len(ENSEMBLE_SEEDS)}_cap{cap_tag}"
    EXPERIMENTS_STAGE2.append({
        'base_name': name,
        'name': exp_name,
        'scheme': 'FP8',
        'manta_n': cfg['manta_n'],
        'pile_n': cfg['pile_n'],
        'arc_n': cfg['arc_n'],
        'pile_mode': cfg.get('pile_mode', 'basic'),
        'seeds': ENSEMBLE_SEEDS,
        'ensemble_cap': ENSEMBLE_CAP,
        'max_seq_length': CALIB_MAX_SEQ_LENGTH,
        'cooldown_sec': STAGE2_COOLDOWN_SEC,
        'run': RUN_STAGE2,
    })

print('Stage-2 experiments:', len(EXPERIMENTS_STAGE2))
for e in EXPERIMENTS_STAGE2:
    total_base = e['manta_n'] + e['pile_n'] + e['arc_n']
    merged_size = total_base * len(e['seeds']) if e['ensemble_cap'] is None else min(total_base * len(e['seeds']), e['ensemble_cap'])
    print(
        f"- {e['name']} | base={e['base_name']} | base_total={total_base} | "
        f"merged={merged_size} | max_seq={e['max_seq_length']} | run={e['run']}"
    )


In [ ]:
if 'EXPERIMENTS_STAGE2' not in globals():
    print('[INFO] EXPERIMENTS_STAGE2가 없습니다. Cell 11(실험 정의)을 먼저 실행하세요.')
    EXPERIMENTS_STAGE2 = []

def hard_cleanup(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()


def run_seed_ensemble_experiment(exp):
    out_dir = BASE_DIR / exp['name'] / 'model'

    print()
    print('=' * 100)
    print(f"SE | {exp['name']}")
    print('=' * 100)

    model = None
    tok = None
    tok_for_calib = None
    calib_ds = None
    recipe = None

    try:
        set_seed(SEED)
        tok_for_calib = AutoTokenizer.from_pretrained(str(BASE_MODEL), trust_remote_code=True)

        calib_ds = build_seed_ensemble_calib(
            tokenizer=tok_for_calib,
            manta_n=exp['manta_n'],
            pile_n=exp['pile_n'],
            arc_n=exp['arc_n'],
            seeds=exp['seeds'],
            pile_mode=exp.get('pile_mode', 'basic'),
            ensemble_cap=exp.get('ensemble_cap'),
            final_shuffle_seed=SEED,
        )
        calib_n = len(calib_ds)

        print(
            f"calib merged: n={calib_n}, seeds={exp['seeds']}, "
            f"base(m,p,a)=({exp['manta_n']},{exp['pile_n']},{exp['arc_n']})"
        )

        model, tok = load_model_and_tokenizer()

        recipe = QuantizationModifier(
            targets='Linear',
            scheme='FP8',
            ignore=BASE_IGNORE,
        )

        t0 = time.time()
        oneshot(
            model=model,
            dataset=calib_ds,
            recipe=recipe,
            max_seq_length=exp.get('max_seq_length', CALIB_MAX_SEQ_LENGTH),
            num_calibration_samples=calib_n,
        )
        print(f"quant done: {time.time() - t0:.1f}s")

        save_model(model, tok, out_dir)
        zip_path = make_submission_zip(out_dir, f"submit_{exp['name']}")

        return {
            'name': exp['name'],
            'base_name': exp['base_name'],
            'model_dir': str(out_dir),
            'zip_path': zip_path,
            'calib_n': calib_n,
            'seeds': exp['seeds'],
        }
    finally:
        hard_cleanup(recipe, model, tok, tok_for_calib, calib_ds)


built_stage2 = []
for exp in EXPERIMENTS_STAGE2:
    if not exp.get('run', False):
        continue

    built_stage2.append(run_seed_ensemble_experiment(exp))

    cooldown = int(exp.get('cooldown_sec', 0) or 0)
    if cooldown > 0:
        print(f"cooldown {cooldown}s before next experiment...")
        time.sleep(cooldown)

print('\nStage-2 built:', len(built_stage2))
for b in built_stage2:
    print('-', b['name'], '| base=', b['base_name'], '| calib_n=', b['calib_n'])


---
## 5. 평가 명령 생성 (raw top3 + seed-ensemble 비교)

In [ ]:
model_paths = [str(BASE_MODEL)]

# 기준 비교를 위해 dynamic + raw top3를 포함
dyn_path = SRC_STAGE1_CODE_DIR / 'exp0_fp8_dynamic_base' / 'model'
if (dyn_path / 'config.json').exists():
    model_paths.append(str(dyn_path))

for name in selected_top:
    p = SRC_STAGE1_CODE_DIR / name / 'model'
    if (p / 'config.json').exists():
        model_paths.append(str(p))

# seed-ensemble 결과 모델
for exp in EXPERIMENTS_STAGE2:
    p = BASE_DIR / exp['name'] / 'model'
    if (p / 'config.json').exists():
        model_paths.append(str(p))

# 중복 제거
seen = set()
model_paths = [x for x in model_paths if not (x in seen or seen.add(x))]

EVAL_CMD = (
    f"python {EVAL_SCRIPT} "
    f"--model_path {' '.join(model_paths)} "
    "--n_runs 3 "
    "--baseline_model_idx 0 "
    "--tasks gsm8k,mmlu,arc_challenge "
    "--limit 512 "
    f"--output_dir {EVAL_OUTPUT_DIR}"
)

print('eval models:', len(model_paths))
print('eval command:')
print(EVAL_CMD)

# 실제 실행 시 주석 해제
# !{EVAL_CMD}

---
## 6. 결과 요약 (재시작/터미널 실행 복귀 대응)

In [ ]:
def _iter_eval_json_with_logs(eval_dirs):
    files = []
    for d in eval_dirs:
        d = Path(d)
        if d.exists():
            files.extend(d.glob('eval_*.json'))
            files.extend(d.glob('comparison_*.json'))

    for d in eval_dirs:
        d = Path(d)
        if not d.exists():
            continue
        for logf in sorted(d.glob('nohup*.log'), key=lambda x: x.stat().st_mtime, reverse=True)[:8]:
            try:
                txt = logf.read_text(encoding='utf-8', errors='ignore')
            except Exception:
                continue
            for m in re.findall(r'결과 저장:\s*(\S*(?:comparison|eval)_\d+\.json)', txt):
                p = Path(m)
                if p.exists():
                    files.append(p)

    uniq = {f.resolve() for f in files if f.exists()}
    return sorted(uniq, key=lambda p: p.stat().st_mtime, reverse=True)


def load_best_eval_summary(eval_dirs, candidate_paths=None):
    candidate_set = {_real(p) for p in (candidate_paths or [])}
    best = None  # overlap, mtime, path, rows

    for f in _iter_eval_json_with_logs(eval_dirs):
        rows = _load_rows(f)
        if rows is None:
            continue

        row_paths = {
            _real(r.get('model_path', ''))
            for r in rows
            if isinstance(r, dict) and r.get('model_path')
        }
        overlap = len(candidate_set & row_paths) if candidate_set else 0
        sc = (overlap, f.stat().st_mtime)
        if best is None or sc > (best[0], best[1]):
            best = (overlap, f.stat().st_mtime, f, rows)

    if best is None:
        return None, None, 0
    return best[2], best[3], best[0]


def summarize_proxy(rows, base_model_path):
    base_real = _real(base_model_path)
    base_row = None
    for r in rows:
        if _real(r.get('model_path', '')) == base_real:
            base_row = r
            break
    if base_row is None:
        raise ValueError('base row not found')

    base_perf = base_row.get('overall_median')
    base_time = (base_row.get('time_stats') or {}).get('median')
    if not _is_num(base_perf) or not _is_num(base_time):
        raise ValueError('base perf/time missing')

    out = []
    for r in rows:
        perf = r.get('overall_median')
        t = (r.get('time_stats') or {}).get('median')
        if not _is_num(perf) or not _is_num(t):
            continue

        pn, sn, sc = compute_competition_score(float(perf), float(base_perf), float(t), float(base_time))
        out.append({
            'model_name': r.get('model_name', 'unknown'),
            'model_path': r.get('model_path'),
            'perf': float(perf),
            'time': float(t),
            'perf_norm': float(pn),
            'speed_norm': float(sn),
            'score_proxy': float(sc),
            'gsm8k': float(r.get('gsm8k_median')) if _is_num(r.get('gsm8k_median')) else None,
            'mmlu': float(r.get('mmlu_median')) if _is_num(r.get('mmlu_median')) else None,
            'arc': float(r.get('arc_challenge_median')) if _is_num(r.get('arc_challenge_median')) else None,
        })

    out.sort(key=lambda x: x['score_proxy'], reverse=True)
    return out


def print_rank(ranked):
    print('rank | model | score_proxy | PerfNorm | SpeedNorm | overall | gsm8k | mmlu | arc | time')
    for i, r in enumerate(ranked, 1):
        print(
            f"{i:>4d} | {r['model_name']:<58} | {_fmt(r['score_proxy'])} | "
            f"{_fmt(r['perf_norm'])} | {_fmt(r['speed_norm'], signed=True)} | {_fmt(r['perf'])} | "
            f"{_fmt(r['gsm8k'])} | {_fmt(r['mmlu'])} | {_fmt(r['arc'])} | {_fmt(r['time'], d=1)}"
        )


def print_raw_vs_ensemble(ranked, selected_raw_names):
    idx = {r['model_name']: r for r in ranked}

    print('\n[Raw vs Seed-Ensemble]')
    for raw in selected_raw_names:
        raw_code = 'code__' + raw
        raw_row = idx.get(raw_code)
        if raw_row is None:
            print('-', raw, ': raw row not found')
            continue

        # 현재 노트북 naming rule 기반
        hits = [r for r in ranked if r['model_name'].startswith('code__exp_se_' + raw)]
        if not hits:
            print('-', raw, ': ensemble row not found')
            continue

        ens = hits[0]
        print(
            f"- {raw}: raw={_fmt(raw_row['score_proxy'])}, ens={_fmt(ens['score_proxy'])}, "
            f"dScore={_fmt(ens['score_proxy'] - raw_row['score_proxy'], signed=True)} | "
            f"dOverall={_fmt(ens['perf'] - raw_row['perf'], signed=True)} | "
            f"dTime={_fmt(ens['time'] - raw_row['time'], d=1, signed=True)}s"
        )


candidate_paths = [BASE_MODEL]
candidate_paths += [SRC_STAGE1_CODE_DIR / 'exp0_fp8_dynamic_base' / 'model']
candidate_paths += [SRC_STAGE1_CODE_DIR / n / 'model' for n in selected_top]
candidate_paths += [BASE_DIR / exp['name'] / 'model' for exp in EXPERIMENTS_STAGE2]

search_dirs = [EVAL_OUTPUT_DIR, ROOT / 'eval', ROOT / '10_third_dataset_search/eval_results']
summary_file, rows, overlap = load_best_eval_summary(search_dirs, candidate_paths)

if summary_file is None:
    print('eval summary를 찾지 못했습니다.')
    print('검색 경로:', [str(x) for x in search_dirs])
else:
    print('summary:', summary_file)
    print(f'candidate overlap: {overlap}/{len(candidate_paths)}')
    ranked = summarize_proxy(rows, str(BASE_MODEL))
    print()
    print_rank(ranked)
    print_raw_vs_ensemble(ranked, selected_top)

---
## 7. ZIP 및 제출 우선 후보 확인

In [ ]:
print('ZIP 목록:')
print('-' * 100)
zips = sorted(glob.glob(str(BASE_DIR / '*/submit_*.zip')))
for z in zips:
    size_mb = os.path.getsize(z) / 1024 / 1024
    print(f'  {z} ({size_mb:.1f} MB)')
if not zips:
    print('  (없음)')

---
## 8. (선택) nohup 실행 명령 생성

In [ ]:
LOG = EVAL_OUTPUT_DIR / f"nohup_run_eval_{time.strftime('%Y%m%d_%H%M%S')}.log"
NOHUP_CMD = f"nohup {EVAL_CMD} > {LOG} 2>&1 & echo PID=$! LOG={LOG}"
print(NOHUP_CMD)